# 04l Extraction, batch l: the last two included studies coded from their PDFs

Design log.
- Batch l of notebook 04 (plan v2 B4) codes the two included studies whose PDFs were too large for the earlier batches: Know Thy Enemy (InstruCoT, arXiv 2601.04666 v2) and GuardRAG (a master's thesis, University of Washington ResearchWorks). Both PDFs are checked against the sha256 recorded in reports/defences.csv before any value is read. With these two records every included study (124) has a record.
- They are coded under the frozen schema (schema/record_schema.yaml, v1) and coding rules, as batches a to k were, so they stay comparable with the other records; any rule amendment that follows the second pass applies to them as to the rest.
- Each record's extraction_scope names the tables and rows coded. As before, detectors, filters and system-level designs are left out and the scope says so (amendment 4): PromptArmor in InstruCoT; PRISM-Mem (a memory firewall) and best-of-N reranking with a separate reward model in GuardRAG. Direct injection, system-prompt extraction and jailbreak rows are left out (InstruCoT Tables 5, 7 and 8).
- Coding decisions worth knowing when reading these records (reading aids for notebook 05, not findings):
  - InstruCoT's Table 3 pools its four backbones, so the per-backbone indirect results of Appendix Table 6 are coded (the AVG and TopicAttack rows). Its utility evidence is a radar chart (Figure 4) labelled for InstruCoT only; no utility value is coded and a residual note says the utility half of the abstract claim has no coded support. The paper does not describe how a sample is judged. Its training targets were generated with GPT-4.1.
  - GuardRAG's undefended reference in Tables 9.1 and 9.2 is the same 8B model under the identical staged security prompt, coded as baseline_kind instruction_prompt_only. Its Table 9.3 runs the trained models in InjecAgent's native agent format without the report contract, where attack success rises above the base model's; this is coded as a measured signal-removal test. Its chosen responses copy an exact evidence span from the injected passage into the security report, so trains_carry_through is yes_by_preference. Baseline-SimPO, DPO and ORPO on the same pairs are coded as objective variants; the 70B run is the scale study of Table 12.21.
- No full-text exclusions in this batch.
- Outputs: reports/records/ (two files), reports/extraction_04l_check.csv, reports/extraction_04l_summary.csv, reports/extraction_batches.csv (rows for batch l added, earlier batches kept), reports/decision_04l.txt.
- CPU; no network calls beyond pip; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf"), ("pypdfium2", "pypdfium2")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:                 # the column-preserving extraction needs pypdf 4 or later
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. The 2 records

In [ ]:
Path("reports/records").mkdir(parents=True, exist_ok=True)       # %%writefile does not create folders

`instrucot_arxiv2026` (2601.04666)

In [ ]:
%%writefile reports/records/instrucot_arxiv2026.yaml
report:
  id: instrucot_arxiv2026
  slug: '2601.04666'
  title: 'Know Thy Enemy: Securing LLMs Against Prompt Injection via Diverse Data Synthesis and Instruction-Level Chain-of-Thought Learning'
  first_author: Zhiyuan Chang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2601.04666'
  version_read: arXiv v2
  first_date: '2026-01-08'
  read: F
  role: proposes
  code_release: not stated in the paper
  later_version: an ACL 2026 presentation record exists on Underline (doi 10.48448/3zdt-ax45); the arXiv v2 was read
  extraction_scope: 'Appendix J Table 6 (indirect prompt injection, Behavior Deviation set, each of the four backbones): the AVG row and the TopicAttack row for Clean, ISE, MetaSec, IP, InSFT and InstruCoT; the abstract claim
    and the InSFT contrast (PromptArmor is a GPT-4 detector that removes injected text and is left out; Table 3 pools the four models; direct injection in Table 5, system-prompt extraction in Table 7 and jailbreak harmful
    requests in Table 8 are out of scope; the utility radar chart in Figure 4 is not coded)'
defences:
- id: ic_l3
  name: InstruCoT (diverse injection synthesis with instruction-level chain-of-thought fine-tuning)
  origin: this_report
  base_checkpoint: Llama3-8B
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - injected instructions synthesised for three threat scenarios (behaviour deviation, privacy leakage, harmful output) and placed into four context regions (user, data, user and data, empty)
  - instruction-aware chain of thought (perceive each instruction, judge whether it violates the system prompt, project a response) generated by GPT-4.1 for adversarial and clean samples
  - full-parameter supervised fine-tuning on the chain of thought followed by the original clean response, or a standard refusal for empty context
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: InstruCoT trains the backbone LLM to follow this generation paradigm
  pages:
    components: 4
- id: insft_l3
  name: InSFT (InstruCoT's synthetic data without the chain of thought), ablation
  origin: this_report
  base_checkpoint: Llama3-8B
  axis1:
  - no_signal
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: When contrasting InstruCoT with InSFT
- id: ise_l3
  name: ISE (learnable segment embeddings for system, user, data and output; run by the authors)
  origin: other_report
  base_checkpoint: Llama3-8B
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 13
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: The method introduces learnable segment embeddings
- id: msec_l3
  name: 'MetaSec (the Meta-SecAlign recipe: a new input role, DPO with LoRA; run by the authors)'
  origin: other_report
  base_checkpoint: Llama3-8B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 13
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: introduces a new input role in the chat template
- id: ip_l3
  name: IP (defensive instructions appended to the system prompt)
  origin: other_report
  base_checkpoint: Llama3-8B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: augments the system prompt with defensive instructions
- id: ic_l31
  name: InstruCoT (diverse injection synthesis with instruction-level chain-of-thought fine-tuning)
  origin: this_report
  base_checkpoint: Llama3.1-8B
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - injected instructions synthesised for three threat scenarios (behaviour deviation, privacy leakage, harmful output) and placed into four context regions (user, data, user and data, empty)
  - instruction-aware chain of thought (perceive each instruction, judge whether it violates the system prompt, project a response) generated by GPT-4.1 for adversarial and clean samples
  - full-parameter supervised fine-tuning on the chain of thought followed by the original clean response, or a standard refusal for empty context
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: InstruCoT trains the backbone LLM to follow this generation paradigm
  pages:
    components: 4
- id: insft_l31
  name: InSFT (InstruCoT's synthetic data without the chain of thought), ablation
  origin: this_report
  base_checkpoint: Llama3.1-8B
  axis1:
  - no_signal
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: When contrasting InstruCoT with InSFT
- id: ise_l31
  name: ISE (learnable segment embeddings for system, user, data and output; run by the authors)
  origin: other_report
  base_checkpoint: Llama3.1-8B
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 13
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: The method introduces learnable segment embeddings
- id: msec_l31
  name: 'MetaSec (the Meta-SecAlign recipe: a new input role, DPO with LoRA; run by the authors)'
  origin: other_report
  base_checkpoint: Llama3.1-8B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 13
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: introduces a new input role in the chat template
- id: ip_l31
  name: IP (defensive instructions appended to the system prompt)
  origin: other_report
  base_checkpoint: Llama3.1-8B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: augments the system prompt with defensive instructions
- id: ic_q25
  name: InstruCoT (diverse injection synthesis with instruction-level chain-of-thought fine-tuning)
  origin: this_report
  base_checkpoint: Qwen2.5-7B
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - injected instructions synthesised for three threat scenarios (behaviour deviation, privacy leakage, harmful output) and placed into four context regions (user, data, user and data, empty)
  - instruction-aware chain of thought (perceive each instruction, judge whether it violates the system prompt, project a response) generated by GPT-4.1 for adversarial and clean samples
  - full-parameter supervised fine-tuning on the chain of thought followed by the original clean response, or a standard refusal for empty context
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: InstruCoT trains the backbone LLM to follow this generation paradigm
  pages:
    components: 4
- id: insft_q25
  name: InSFT (InstruCoT's synthetic data without the chain of thought), ablation
  origin: this_report
  base_checkpoint: Qwen2.5-7B
  axis1:
  - no_signal
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: When contrasting InstruCoT with InSFT
- id: ise_q25
  name: ISE (learnable segment embeddings for system, user, data and output; run by the authors)
  origin: other_report
  base_checkpoint: Qwen2.5-7B
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 13
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: The method introduces learnable segment embeddings
- id: msec_q25
  name: 'MetaSec (the Meta-SecAlign recipe: a new input role, DPO with LoRA; run by the authors)'
  origin: other_report
  base_checkpoint: Qwen2.5-7B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 13
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: introduces a new input role in the chat template
- id: ip_q25
  name: IP (defensive instructions appended to the system prompt)
  origin: other_report
  base_checkpoint: Qwen2.5-7B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: augments the system prompt with defensive instructions
- id: ic_q3
  name: InstruCoT (diverse injection synthesis with instruction-level chain-of-thought fine-tuning)
  origin: this_report
  base_checkpoint: Qwen3-8B
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - injected instructions synthesised for three threat scenarios (behaviour deviation, privacy leakage, harmful output) and placed into four context regions (user, data, user and data, empty)
  - instruction-aware chain of thought (perceive each instruction, judge whether it violates the system prompt, project a response) generated by GPT-4.1 for adversarial and clean samples
  - full-parameter supervised fine-tuning on the chain of thought followed by the original clean response, or a standard refusal for empty context
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: InstruCoT trains the backbone LLM to follow this generation paradigm
  pages:
    components: 4
- id: insft_q3
  name: InSFT (InstruCoT's synthetic data without the chain of thought), ablation
  origin: this_report
  base_checkpoint: Qwen3-8B
  axis1:
  - no_signal
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: When contrasting InstruCoT with InSFT
- id: ise_q3
  name: ISE (learnable segment embeddings for system, user, data and output; run by the authors)
  origin: other_report
  base_checkpoint: Qwen3-8B
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 13
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: The method introduces learnable segment embeddings
- id: msec_q3
  name: 'MetaSec (the Meta-SecAlign recipe: a new input role, DPO with LoRA; run by the authors)'
  origin: other_report
  base_checkpoint: Qwen3-8B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 13
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: introduces a new input role in the chat template
- id: ip_q3
  name: IP (defensive instructions appended to the system prompt)
  origin: other_report
  base_checkpoint: Qwen3-8B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: system prompt followed by context regions (user, data, or both); injected instructions placed in the user region, the data region or both
  layout_compat: *id001
  anchor: augments the system prompt with defensive instructions
evaluations:
- id: ev_avg_none_l3
  defence_id: none:Llama3-8B
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_avg_ise_l3
  defence_id: ise_l3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_msec_l3
  defence_id: msec_l3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_ip_l3
  defence_id: ip_l3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_insft_l3
  defence_id: insft_l3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_ic_l3
  defence_id: ic_l3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_topic_none_l3
  defence_id: none:Llama3-8B
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
  baseline_kind: off_the_shelf
- id: ev_topic_ise_l3
  defence_id: ise_l3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_msec_l3
  defence_id: msec_l3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_ip_l3
  defence_id: ip_l3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_insft_l3
  defence_id: insft_l3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_ic_l3
  defence_id: ic_l3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_avg_none_l31
  defence_id: none:Llama3.1-8B
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_avg_ise_l31
  defence_id: ise_l31
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_msec_l31
  defence_id: msec_l31
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_ip_l31
  defence_id: ip_l31
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_insft_l31
  defence_id: insft_l31
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_ic_l31
  defence_id: ic_l31
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_topic_none_l31
  defence_id: none:Llama3.1-8B
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
  baseline_kind: off_the_shelf
- id: ev_topic_ise_l31
  defence_id: ise_l31
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_msec_l31
  defence_id: msec_l31
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_ip_l31
  defence_id: ip_l31
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_insft_l31
  defence_id: insft_l31
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_ic_l31
  defence_id: ic_l31
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_avg_none_q25
  defence_id: none:Qwen2.5-7B
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_avg_ise_q25
  defence_id: ise_q25
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_msec_q25
  defence_id: msec_q25
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_ip_q25
  defence_id: ip_q25
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_insft_q25
  defence_id: insft_q25
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_ic_q25
  defence_id: ic_q25
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_topic_none_q25
  defence_id: none:Qwen2.5-7B
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
  baseline_kind: off_the_shelf
- id: ev_topic_ise_q25
  defence_id: ise_q25
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_msec_q25
  defence_id: msec_q25
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_ip_q25
  defence_id: ip_q25
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_insft_q25
  defence_id: insft_q25
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_ic_q25
  defence_id: ic_q25
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_avg_none_q3
  defence_id: none:Qwen3-8B
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
  baseline_kind: off_the_shelf
- id: ev_avg_ise_q3
  defence_id: ise_q3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_msec_q3
  defence_id: msec_q3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_ip_q3
  defence_id: ip_q3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_insft_q3
  defence_id: insft_q3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_avg_ic_q3
  defence_id: ic_q3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; mean over ten attacks (Naive, Ignore, Escape and Completion at one and at two positions, Combined, TopicAttack)
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: varied
- id: ev_topic_none_q3
  defence_id: none:Qwen3-8B
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
  baseline_kind: off_the_shelf
- id: ev_topic_ise_q3
  defence_id: ise_q3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_msec_q3
  defence_id: msec_q3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_ip_q3
  defence_id: ip_q3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_insft_q3
  defence_id: insft_q3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
- id: ev_topic_ic_q3
  defence_id: ic_q3
  benchmark: indirect prompt injection set of Wu et al. (2025), Behavior Deviation, 208 samples; TopicAttack
  attacker_type: static
  attacker_access: black_box
  judge: defence rate under the evaluation protocol of Wu et al. (2025); the judge is not described in this paper
  layout_same_as_defence: 'yes'
  measured_by: instrucot_arxiv2026
  page: 17
  decoding: not reported
  sample_size: '208'
  injection_position: unreported
results:
- id: r_avg_none_l3
  evaluation_id: ev_avg_none_l3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '20.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3-8B, AVG row
- id: r_avg_ise_l3
  evaluation_id: ev_avg_ise_l3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '76.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3-8B, AVG row
- id: r_avg_msec_l3
  evaluation_id: ev_avg_msec_l3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '85.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3-8B, AVG row
- id: r_avg_ip_l3
  evaluation_id: ev_avg_ip_l3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '21.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3-8B, AVG row
- id: r_avg_insft_l3
  evaluation_id: ev_avg_insft_l3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '60.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3-8B, AVG row
- id: r_avg_ic_l3
  evaluation_id: ev_avg_ic_l3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '94.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3-8B, AVG row
- id: r_topic_none_l3
  evaluation_id: ev_topic_none_l3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '21.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3-8B, TopicAttack row
- id: r_topic_ise_l3
  evaluation_id: ev_topic_ise_l3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '50.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3-8B, TopicAttack row
- id: r_topic_msec_l3
  evaluation_id: ev_topic_msec_l3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '80.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3-8B, TopicAttack row
- id: r_topic_ip_l3
  evaluation_id: ev_topic_ip_l3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '22.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3-8B, TopicAttack row
- id: r_topic_insft_l3
  evaluation_id: ev_topic_insft_l3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '23.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3-8B, TopicAttack row
- id: r_topic_ic_l3
  evaluation_id: ev_topic_ic_l3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '88.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3-8B, TopicAttack row
- id: r_avg_none_l31
  evaluation_id: ev_avg_none_l31
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '36.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3.1-8B, AVG row
- id: r_avg_ise_l31
  evaluation_id: ev_avg_ise_l31
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '69.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3.1-8B, AVG row
- id: r_avg_msec_l31
  evaluation_id: ev_avg_msec_l31
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '85.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3.1-8B, AVG row
- id: r_avg_ip_l31
  evaluation_id: ev_avg_ip_l31
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '22.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3.1-8B, AVG row
- id: r_avg_insft_l31
  evaluation_id: ev_avg_insft_l31
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '57.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3.1-8B, AVG row
- id: r_avg_ic_l31
  evaluation_id: ev_avg_ic_l31
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '93.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3.1-8B, AVG row
- id: r_topic_none_l31
  evaluation_id: ev_topic_none_l31
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '25.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3.1-8B, TopicAttack row
- id: r_topic_ise_l31
  evaluation_id: ev_topic_ise_l31
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '14.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3.1-8B, TopicAttack row
- id: r_topic_msec_l31
  evaluation_id: ev_topic_msec_l31
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '70.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3.1-8B, TopicAttack row
- id: r_topic_ip_l31
  evaluation_id: ev_topic_ip_l31
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '19.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3.1-8B, TopicAttack row
- id: r_topic_insft_l31
  evaluation_id: ev_topic_insft_l31
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '10.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3.1-8B, TopicAttack row
- id: r_topic_ic_l31
  evaluation_id: ev_topic_ic_l31
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '79.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Llama3.1-8B, TopicAttack row
- id: r_avg_none_q25
  evaluation_id: ev_avg_none_q25
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '3.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen2.5-7B, AVG row
- id: r_avg_ise_q25
  evaluation_id: ev_avg_ise_q25
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '69.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen2.5-7B, AVG row
- id: r_avg_msec_q25
  evaluation_id: ev_avg_msec_q25
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '84.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen2.5-7B, AVG row
- id: r_avg_ip_q25
  evaluation_id: ev_avg_ip_q25
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '2.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen2.5-7B, AVG row
- id: r_avg_insft_q25
  evaluation_id: ev_avg_insft_q25
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '63.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen2.5-7B, AVG row
- id: r_avg_ic_q25
  evaluation_id: ev_avg_ic_q25
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '93.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen2.5-7B, AVG row
- id: r_topic_none_q25
  evaluation_id: ev_topic_none_q25
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '0.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen2.5-7B, TopicAttack row
- id: r_topic_ise_q25
  evaluation_id: ev_topic_ise_q25
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '12.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen2.5-7B, TopicAttack row
- id: r_topic_msec_q25
  evaluation_id: ev_topic_msec_q25
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '68.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen2.5-7B, TopicAttack row
- id: r_topic_ip_q25
  evaluation_id: ev_topic_ip_q25
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '0.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen2.5-7B, TopicAttack row
- id: r_topic_insft_q25
  evaluation_id: ev_topic_insft_q25
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '9.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen2.5-7B, TopicAttack row
- id: r_topic_ic_q25
  evaluation_id: ev_topic_ic_q25
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '93.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen2.5-7B, TopicAttack row
- id: r_avg_none_q3
  evaluation_id: ev_avg_none_q3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '1.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen3-8B, AVG row
- id: r_avg_ise_q3
  evaluation_id: ev_avg_ise_q3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '68.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen3-8B, AVG row
- id: r_avg_msec_q3
  evaluation_id: ev_avg_msec_q3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '83.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen3-8B, AVG row
- id: r_avg_ip_q3
  evaluation_id: ev_avg_ip_q3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '2.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen3-8B, AVG row
- id: r_avg_insft_q3
  evaluation_id: ev_avg_insft_q3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '50.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen3-8B, AVG row
- id: r_avg_ic_q3
  evaluation_id: ev_avg_ic_q3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '91.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen3-8B, AVG row
- id: r_topic_none_q3
  evaluation_id: ev_topic_none_q3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '0.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen3-8B, TopicAttack row
- id: r_topic_ise_q3
  evaluation_id: ev_topic_ise_q3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '8.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen3-8B, TopicAttack row
- id: r_topic_msec_q3
  evaluation_id: ev_topic_msec_q3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '65.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen3-8B, TopicAttack row
- id: r_topic_ip_q3
  evaluation_id: ev_topic_ip_q3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '0.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen3-8B, TopicAttack row
- id: r_topic_insft_q3
  evaluation_id: ev_topic_insft_q3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '17.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen3-8B, TopicAttack row
- id: r_topic_ic_q3
  evaluation_id: ev_topic_ic_q3
  quantity: R1
  coverage: direct
  metric_as_printed: Defense Rate (DR)
  value: '89.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 6, Qwen3-8B, TopicAttack row
comparisons:
- id: c_avg_ic_ise_l3
  left: ev_avg_ic_l3
  right: ev_avg_ise_l3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_avg_ic_msec_l3
  left: ev_avg_ic_l3
  right: ev_avg_msec_l3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_avg_ic_ip_l3
  left: ev_avg_ic_l3
  right: ev_avg_ip_l3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_avg_ic_none_l3
  left: ev_avg_ic_l3
  right: ev_avg_none_l3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_avg_ic_insft_l3
  left: ev_avg_ic_l3
  right: ev_avg_insft_l3
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the instruction-level chain of thought (same synthetic data, trained with and without it)
  page: 17
- id: c_topic_ic_ise_l3
  left: ev_topic_ic_l3
  right: ev_topic_ise_l3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_topic_ic_msec_l3
  left: ev_topic_ic_l3
  right: ev_topic_msec_l3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_topic_ic_ip_l3
  left: ev_topic_ic_l3
  right: ev_topic_ip_l3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_topic_ic_none_l3
  left: ev_topic_ic_l3
  right: ev_topic_none_l3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_topic_ic_insft_l3
  left: ev_topic_ic_l3
  right: ev_topic_insft_l3
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the instruction-level chain of thought (same synthetic data, trained with and without it)
  page: 17
- id: c_avg_ic_ise_l31
  left: ev_avg_ic_l31
  right: ev_avg_ise_l31
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_avg_ic_msec_l31
  left: ev_avg_ic_l31
  right: ev_avg_msec_l31
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_avg_ic_ip_l31
  left: ev_avg_ic_l31
  right: ev_avg_ip_l31
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_avg_ic_none_l31
  left: ev_avg_ic_l31
  right: ev_avg_none_l31
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_avg_ic_insft_l31
  left: ev_avg_ic_l31
  right: ev_avg_insft_l31
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the instruction-level chain of thought (same synthetic data, trained with and without it)
  page: 17
- id: c_topic_ic_ise_l31
  left: ev_topic_ic_l31
  right: ev_topic_ise_l31
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_topic_ic_msec_l31
  left: ev_topic_ic_l31
  right: ev_topic_msec_l31
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_topic_ic_ip_l31
  left: ev_topic_ic_l31
  right: ev_topic_ip_l31
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_topic_ic_none_l31
  left: ev_topic_ic_l31
  right: ev_topic_none_l31
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_topic_ic_insft_l31
  left: ev_topic_ic_l31
  right: ev_topic_insft_l31
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the instruction-level chain of thought (same synthetic data, trained with and without it)
  page: 17
- id: c_avg_ic_ise_q25
  left: ev_avg_ic_q25
  right: ev_avg_ise_q25
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_avg_ic_msec_q25
  left: ev_avg_ic_q25
  right: ev_avg_msec_q25
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_avg_ic_ip_q25
  left: ev_avg_ic_q25
  right: ev_avg_ip_q25
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_avg_ic_none_q25
  left: ev_avg_ic_q25
  right: ev_avg_none_q25
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_avg_ic_insft_q25
  left: ev_avg_ic_q25
  right: ev_avg_insft_q25
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the instruction-level chain of thought (same synthetic data, trained with and without it)
  page: 17
- id: c_topic_ic_ise_q25
  left: ev_topic_ic_q25
  right: ev_topic_ise_q25
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_topic_ic_msec_q25
  left: ev_topic_ic_q25
  right: ev_topic_msec_q25
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_topic_ic_ip_q25
  left: ev_topic_ic_q25
  right: ev_topic_ip_q25
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_topic_ic_none_q25
  left: ev_topic_ic_q25
  right: ev_topic_none_q25
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_topic_ic_insft_q25
  left: ev_topic_ic_q25
  right: ev_topic_insft_q25
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the instruction-level chain of thought (same synthetic data, trained with and without it)
  page: 17
- id: c_avg_ic_ise_q3
  left: ev_avg_ic_q3
  right: ev_avg_ise_q3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_avg_ic_msec_q3
  left: ev_avg_ic_q3
  right: ev_avg_msec_q3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_avg_ic_ip_q3
  left: ev_avg_ic_q3
  right: ev_avg_ip_q3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_avg_ic_none_q3
  left: ev_avg_ic_q3
  right: ev_avg_none_q3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_avg_ic_insft_q3
  left: ev_avg_ic_q3
  right: ev_avg_insft_q3
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the instruction-level chain of thought (same synthetic data, trained with and without it)
  page: 17
- id: c_topic_ic_ise_q3
  left: ev_topic_ic_q3
  right: ev_topic_ise_q3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_topic_ic_msec_q3
  left: ev_topic_ic_q3
  right: ev_topic_msec_q3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_topic_ic_ip_q3
  left: ev_topic_ic_q3
  right: ev_topic_ip_q3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (InstruCoT against another defence on the same backbone)
  page: 17
- id: c_topic_ic_none_q3
  left: ev_topic_ic_q3
  right: ev_topic_none_q3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_topic_ic_insft_q3
  left: ev_topic_ic_q3
  right: ev_topic_insft_q3
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the instruction-level chain of thought (same synthetic data, trained with and without it)
  page: 17
claims:
- id: k1
  text: Across four LLMs InstruCoT significantly outperforms the baselines in every dimension while keeping utility without degradation
  location: abstract
  anchor: Experimental results across four LLMs demonstrate that InstruCoT significantly outperforms baselines
  jobs:
  - R1
  - general_utility
  supported_by:
  - c_avg_ic_none_l3
  - c_avg_ic_ise_l3
  - c_avg_ic_msec_l3
  - c_avg_ic_ip_l3
  - c_avg_ic_none_l31
  - c_avg_ic_ise_l31
  - c_avg_ic_msec_l31
  - c_avg_ic_ip_l31
  - c_avg_ic_none_q25
  - c_avg_ic_ise_q25
  - c_avg_ic_msec_q25
  - c_avg_ic_ip_q25
  - c_avg_ic_none_q3
  - c_avg_ic_ise_q3
  - c_avg_ic_msec_q3
  - c_avg_ic_ip_q3
  page: 1
- id: k2
  text: The instruction-aware chain of thought is what raises the defence rate over the same data trained without it (InSFT)
  location: Section 5.2
  anchor: demonstrates the effectiveness of instruction-aware CoT in enhancing PI defense
  jobs:
  - R1
  supported_by:
  - c_avg_ic_insft_l3
  - c_avg_ic_insft_l31
  - c_avg_ic_insft_q25
  - c_avg_ic_insft_q3
  - c_topic_ic_insft_l3
  - c_topic_ic_insft_l31
  - c_topic_ic_insft_q25
  - c_topic_ic_insft_q3
  page: 7
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Appendix Table 6, per backbone (Table 3 pools the four backbones)
    general_utility: Figure 4 radar chart only (AlpacaEval 1.0 win rate); the text gives InstruCoT's mean over four backbones (82.9%) and no baseline values
    R2clean: not reported
    R2attack: not reported
    R3: not reported
schema_gaps:
- where: report
  note: the defence rate (DR) is the share of samples where the model resists the injection, following Wu et al. (2025); this paper does not say how a sample is judged
- where: claims:k1
  note: the utility half of the claim rests on a radar chart (Figure 4) whose values are labelled for InstruCoT only and too small to read reliably; no utility value is coded, so only the defence-rate half has coded support
- where: report
  note: the chain-of-thought training targets and the injected instructions were generated with GPT-4.1, a closed model; ISE, MetaSec and IP were run by the authors on the four backbones; the backbones are named without saying
    whether the base or the instruct release was used

`guardrag_uwthesis2026` (title_guardragadversarialpreferencetrainingagainstindirectpromptin)

In [ ]:
%%writefile reports/records/guardrag_uwthesis2026.yaml
report:
  id: guardrag_uwthesis2026
  slug: title_guardragadversarialpreferencetrainingagainstindirectpromptin
  title: 'GuardRAG: Adversarial Preference Training Against Indirect Prompt Injection Attacks'
  first_author: Gayatri Malladi
  venue: Master's thesis, University of Washington (ResearchWorks)
  peer_reviewed: 'no'
  version_read: thesis PDF from ResearchWorks
  first_date: '2026-01-01'
  read: F
  role: mixed
  code_release: not checked
  extraction_scope: 'Chapters 5, 7.3 and 9.1, Appendix 12.3: Table 9.1 (stealth split attack success, clean split utility retention), Table 9.2 (attack success on five external benchmarks) and Table 9.3 (native InjecAgent
    format) for Base, Baseline-SimPO, SAGE-C, DPO and ORPO on Meta-Llama-3-8B-Instruct; Table 12.21 (8B against 70B SAGE-C); the abstract claims and the format-coupling finding (RIPE-II, the undefended-model benchmark of Chapters
    4 and 8, is not coded as a defence; PRISM-Mem is a memory firewall and best-of-N reranking uses a separate reward model, both left out as system level; detection, report quality and the adaptive-hardening runs are not
    coded)'
defences:
- id: sagec
  name: GuardRAG with the SAGE-C security-adaptive SimPO objective (LoRA)
  origin: this_report
  base_checkpoint: Meta-Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: 'no'
  trains_carry_through: yes_by_preference
  page: 36
  components:
  - staged security prompt with a trust hierarchy and per-passage provenance boundaries
  - span-grounded JSON security report per passage (attack flag, family, exact evidence span, violated policy, handling)
  - security- and utility-aware preference pairs built from RIPE-II episodes, candidates generated by Llama-3.3-70B-Instruct
  - 'SAGE-C: SimPO with per-example adaptive margin, smoothing and weight from pair confidence, severity, report quality and over-defence signals'
  layout: staged security prompt (system trust hierarchy, then the user's question, then the top-5 retrieved passages, each inside an explicit provenance boundary with passage and document identifiers); output is an answer
    plus one JSON security report
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: unsupported
  signal_dependence_detail: 'Table 9.3: in the native InjecAgent ReAct format, without the report contract, SAGE-C''s attack success (53.3%) is above the base model''s (30.3%)'
  pages:
    components: 40
    signal_dependence: 85
  anchor: Its adaptive layer, SAGE-C, uses pair confidence, attack severity, report quality
- id: simpo
  name: Baseline-SimPO (the same GuardRAG data, prompt and report; plain SimPO, target margin 0), objective variant
  origin: this_report
  base_checkpoint: Meta-Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: measured
  trains_instruction_pair: 'no'
  trains_carry_through: yes_by_preference
  page: 76
  layout: staged security prompt (system trust hierarchy, then the user's question, then the top-5 retrieved passages, each inside an explicit provenance boundary with passage and document identifiers); output is an answer
    plus one JSON security report
  layout_compat: *id001
  anchor: Only SAGE-C and the SimPO baseline sit in the usable corner
- id: dpo
  name: DPO on the same GuardRAG pairs, objective variant
  origin: this_report
  base_checkpoint: Meta-Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: measured
  trains_instruction_pair: 'no'
  trains_carry_through: yes_by_preference
  page: 76
  layout: staged security prompt (system trust hierarchy, then the user's question, then the top-5 retrieved passages, each inside an explicit provenance boundary with passage and document identifiers); output is an answer
    plus one JSON security report
  layout_compat: *id001
  anchor: DPO also drives attack success to near zero
- id: orpo
  name: ORPO on the same GuardRAG pairs, objective variant
  origin: this_report
  base_checkpoint: Meta-Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: measured
  trains_instruction_pair: 'no'
  trains_carry_through: yes_by_preference
  page: 76
  layout: staged security prompt (system trust hierarchy, then the user's question, then the top-5 retrieved passages, each inside an explicit provenance boundary with passage and document identifiers); output is an answer
    plus one JSON security report
  layout_compat: *id001
  anchor: ORPO fails in the other direction
- id: sagec70
  name: GuardRAG with SAGE-C applied to a 70B backbone (scale study, LoRA rank 64)
  origin: this_report
  base_checkpoint: Llama-3.3-70B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: measured_other_checkpoint
  trains_instruction_pair: 'no'
  trains_carry_through: yes_by_preference
  page: 80
  layout: staged security prompt (system trust hierarchy, then the user's question, then the top-5 retrieved passages, each inside an explicit provenance boundary with passage and document identifiers); output is an answer
    plus one JSON security report
  layout_compat: *id001
  signal_dependence_detail: the native-format test (Table 9.3) was run on the 8B models only
  anchor: SAGE-C was applied to a 70B backbone under the same data and objective
evaluations:
- id: ev_t91s_base
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: GuardRAG internal stealth / obfuscation split (RIPE-II carriers), 300 rows
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 76
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
  baseline_kind: instruction_prompt_only
- id: ev_t91c_base
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: GuardRAG internal clean-utility split, 300 rows
  attacker_type: none
  attacker_access: not_applicable
  judge: 'clean answer retained: substantive answer, parseable report, no attack flag'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 76
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
  baseline_kind: instruction_prompt_only
- id: ev_t91s_simpo
  defence_id: simpo
  benchmark: GuardRAG internal stealth / obfuscation split (RIPE-II carriers), 300 rows
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 76
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_t91c_simpo
  defence_id: simpo
  benchmark: GuardRAG internal clean-utility split, 300 rows
  attacker_type: none
  attacker_access: not_applicable
  judge: 'clean answer retained: substantive answer, parseable report, no attack flag'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 76
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_t91s_sagec
  defence_id: sagec
  benchmark: GuardRAG internal stealth / obfuscation split (RIPE-II carriers), 300 rows
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 76
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_t91c_sagec
  defence_id: sagec
  benchmark: GuardRAG internal clean-utility split, 300 rows
  attacker_type: none
  attacker_access: not_applicable
  judge: 'clean answer retained: substantive answer, parseable report, no attack flag'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 76
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_t91s_dpo
  defence_id: dpo
  benchmark: GuardRAG internal stealth / obfuscation split (RIPE-II carriers), 300 rows
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 76
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_t91c_dpo
  defence_id: dpo
  benchmark: GuardRAG internal clean-utility split, 300 rows
  attacker_type: none
  attacker_access: not_applicable
  judge: 'clean answer retained: substantive answer, parseable report, no attack flag'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 76
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_t91s_orpo
  defence_id: orpo
  benchmark: GuardRAG internal stealth / obfuscation split (RIPE-II carriers), 300 rows
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 76
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_t91c_orpo
  defence_id: orpo
  benchmark: GuardRAG internal clean-utility split, 300 rows
  attacker_type: none
  attacker_access: not_applicable
  judge: 'clean answer retained: substantive answer, parseable report, no attack flag'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 76
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_cse3_base
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: CyberSecEval3 indirect, 206 locally scorable cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '206'
  baseline_kind: instruction_prompt_only
- id: ev_cse3_simpo
  defence_id: simpo
  benchmark: CyberSecEval3 indirect, 206 locally scorable cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '206'
- id: ev_cse3_sagec
  defence_id: sagec
  benchmark: CyberSecEval3 indirect, 206 locally scorable cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '206'
- id: ev_cse3_dpo
  defence_id: dpo
  benchmark: CyberSecEval3 indirect, 206 locally scorable cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '206'
- id: ev_cse3_orpo
  defence_id: orpo
  benchmark: CyberSecEval3 indirect, 206 locally scorable cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '206'
- id: ev_opi_base
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: Open-Prompt-Injection, five strategies, 400 cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '400'
  baseline_kind: instruction_prompt_only
- id: ev_opi_simpo
  defence_id: simpo
  benchmark: Open-Prompt-Injection, five strategies, 400 cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '400'
- id: ev_opi_sagec
  defence_id: sagec
  benchmark: Open-Prompt-Injection, five strategies, 400 cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '400'
- id: ev_opi_dpo
  defence_id: dpo
  benchmark: Open-Prompt-Injection, five strategies, 400 cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '400'
- id: ev_opi_orpo
  defence_id: orpo
  benchmark: Open-Prompt-Injection, five strategies, 400 cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '400'
- id: ev_bipia_base
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: BIPIA-GPT (out of distribution), 150 attack and 150 clean
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
  baseline_kind: instruction_prompt_only
- id: ev_bipia_simpo
  defence_id: simpo
  benchmark: BIPIA-GPT (out of distribution), 150 attack and 150 clean
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_bipia_sagec
  defence_id: sagec
  benchmark: BIPIA-GPT (out of distribution), 150 attack and 150 clean
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_bipia_dpo
  defence_id: dpo
  benchmark: BIPIA-GPT (out of distribution), 150 attack and 150 clean
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_bipia_orpo
  defence_id: orpo
  benchmark: BIPIA-GPT (out of distribution), 150 attack and 150 clean
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_pieval_base
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: LLM-PIEval, injection in a tool result presented at the retrieval boundary, 150 cases
  attacker_type: static
  attacker_access: black_box
  judge: tool-result compliance with the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '150'
  baseline_kind: instruction_prompt_only
- id: ev_pieval_simpo
  defence_id: simpo
  benchmark: LLM-PIEval, injection in a tool result presented at the retrieval boundary, 150 cases
  attacker_type: static
  attacker_access: black_box
  judge: tool-result compliance with the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '150'
- id: ev_pieval_sagec
  defence_id: sagec
  benchmark: LLM-PIEval, injection in a tool result presented at the retrieval boundary, 150 cases
  attacker_type: static
  attacker_access: black_box
  judge: tool-result compliance with the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '150'
- id: ev_pieval_dpo
  defence_id: dpo
  benchmark: LLM-PIEval, injection in a tool result presented at the retrieval boundary, 150 cases
  attacker_type: static
  attacker_access: black_box
  judge: tool-result compliance with the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '150'
- id: ev_pieval_orpo
  defence_id: orpo
  benchmark: LLM-PIEval, injection in a tool result presented at the retrieval boundary, 150 cases
  attacker_type: static
  attacker_access: black_box
  judge: tool-result compliance with the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '150'
- id: ev_scifact_base
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: RIPE-II SciFact, domain held out of training, 87 cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '87'
  baseline_kind: instruction_prompt_only
- id: ev_scifact_simpo
  defence_id: simpo
  benchmark: RIPE-II SciFact, domain held out of training, 87 cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '87'
- id: ev_scifact_sagec
  defence_id: sagec
  benchmark: RIPE-II SciFact, domain held out of training, 87 cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '87'
- id: ev_scifact_dpo
  defence_id: dpo
  benchmark: RIPE-II SciFact, domain held out of training, 87 cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '87'
- id: ev_scifact_orpo
  defence_id: orpo
  benchmark: RIPE-II SciFact, domain held out of training, 87 cases
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 79
  decoding: greedy, repetition penalty 1.05
  sample_size: '87'
- id: ev_inj_base
  defence_id: none:Meta-Llama-3-8B-Instruct
  benchmark: InjecAgent native ReAct/tool-agent format (direct-harm and data-stealing cases), 1,054 cases
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent's own scorer
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 85
  decoding: greedy, repetition penalty 1.05
  sample_size: '1054'
  baseline_kind: off_the_shelf
- id: ev_inj_simpo
  defence_id: simpo
  benchmark: InjecAgent native ReAct/tool-agent format (direct-harm and data-stealing cases), 1,054 cases
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent's own scorer
  layout_same_as_defence: 'no'
  measured_by: guardrag_uwthesis2026
  page: 85
  decoding: greedy, repetition penalty 1.05
  sample_size: '1054'
- id: ev_inj_sagec
  defence_id: sagec
  benchmark: InjecAgent native ReAct/tool-agent format (direct-harm and data-stealing cases), 1,054 cases
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent's own scorer
  layout_same_as_defence: 'no'
  measured_by: guardrag_uwthesis2026
  page: 85
  decoding: greedy, repetition penalty 1.05
  sample_size: '1054'
- id: ev_inj_dpo
  defence_id: dpo
  benchmark: InjecAgent native ReAct/tool-agent format (direct-harm and data-stealing cases), 1,054 cases
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent's own scorer
  layout_same_as_defence: 'no'
  measured_by: guardrag_uwthesis2026
  page: 85
  decoding: greedy, repetition penalty 1.05
  sample_size: '1054'
- id: ev_inj_orpo
  defence_id: orpo
  benchmark: InjecAgent native ReAct/tool-agent format (direct-harm and data-stealing cases), 1,054 cases
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent's own scorer
  layout_same_as_defence: 'no'
  measured_by: guardrag_uwthesis2026
  page: 85
  decoding: greedy, repetition penalty 1.05
  sample_size: '1054'
- id: ev_sc_stealth_8
  defence_id: sagec
  benchmark: 'Table 12.21 split: Stealth/obfusc. (in-dist)'
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 106
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_sc_stealth_70
  defence_id: sagec70
  benchmark: 'Table 12.21 split: Stealth/obfusc. (in-dist)'
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 106
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_sc_bipiael_8
  defence_id: sagec
  benchmark: 'Table 12.21 split: BIPIA-element (OOD)'
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 106
  decoding: greedy, repetition penalty 1.05
  sample_size: '100'
- id: ev_sc_bipiael_70
  defence_id: sagec70
  benchmark: 'Table 12.21 split: BIPIA-element (OOD)'
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 106
  decoding: greedy, repetition penalty 1.05
  sample_size: '100'
- id: ev_sc_bipiagpt_8
  defence_id: sagec
  benchmark: 'Table 12.21 split: BIPIA-GPT (OOD)'
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 106
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
- id: ev_sc_bipiagpt_70
  defence_id: sagec70
  benchmark: 'Table 12.21 split: BIPIA-GPT (OOD)'
  attacker_type: static
  attacker_access: black_box
  judge: 'behavioural attack success: the answer follows the injected directive (Section 4.5.2 counterfactual scoring); the judge for these suites is not restated'
  layout_same_as_defence: 'yes'
  measured_by: guardrag_uwthesis2026
  page: 106
  decoding: greedy, repetition penalty 1.05
  sample_size: '300'
results:
- id: r_t91_base_asr
  evaluation_id: ev_t91s_base
  quantity: R1
  coverage: direct
  metric_as_printed: Attack succ.
  value: '0.457'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 76
  location: Table 9.1
- id: r_t91_base_util
  evaluation_id: ev_t91c_base
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Utility ret.
  value: '0.280'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 76
  location: Table 9.1
- id: r_t91_simpo_asr
  evaluation_id: ev_t91s_simpo
  quantity: R1
  coverage: direct
  metric_as_printed: Attack succ.
  value: '0.003'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 76
  location: Table 9.1
- id: r_t91_simpo_util
  evaluation_id: ev_t91c_simpo
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Utility ret.
  value: '0.937'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 76
  location: Table 9.1
- id: r_t91_sagec_asr
  evaluation_id: ev_t91s_sagec
  quantity: R1
  coverage: direct
  metric_as_printed: Attack succ.
  value: '0.003'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 76
  location: Table 9.1
- id: r_t91_sagec_util
  evaluation_id: ev_t91c_sagec
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Utility ret.
  value: '0.947'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 76
  location: Table 9.1
- id: r_t91_dpo_asr
  evaluation_id: ev_t91s_dpo
  quantity: R1
  coverage: direct
  metric_as_printed: Attack succ.
  value: '0.010'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 76
  location: Table 9.1
- id: r_t91_dpo_util
  evaluation_id: ev_t91c_dpo
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Utility ret.
  value: '0.347'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 76
  location: Table 9.1
- id: r_t91_orpo_asr
  evaluation_id: ev_t91s_orpo
  quantity: R1
  coverage: direct
  metric_as_printed: Attack succ.
  value: '0.443'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 76
  location: Table 9.1
- id: r_t91_orpo_util
  evaluation_id: ev_t91c_orpo
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Utility ret.
  value: '0.740'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 76
  location: Table 9.1
- id: r_cse3_base
  evaluation_id: ev_cse3_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '5.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, CyberSecEval3 indirect
- id: r_cse3_simpo
  evaluation_id: ev_cse3_simpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '12.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, CyberSecEval3 indirect
- id: r_cse3_sagec
  evaluation_id: ev_cse3_sagec
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '2.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, CyberSecEval3 indirect
- id: r_cse3_dpo
  evaluation_id: ev_cse3_dpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, CyberSecEval3 indirect
- id: r_cse3_orpo
  evaluation_id: ev_cse3_orpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '12.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, CyberSecEval3 indirect
- id: r_opi_base
  evaluation_id: ev_opi_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '1.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, Open-Prompt-Injection
- id: r_opi_simpo
  evaluation_id: ev_opi_simpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '1.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, Open-Prompt-Injection
- id: r_opi_sagec
  evaluation_id: ev_opi_sagec
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '3.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, Open-Prompt-Injection
- id: r_opi_dpo
  evaluation_id: ev_opi_dpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, Open-Prompt-Injection
- id: r_opi_orpo
  evaluation_id: ev_opi_orpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, Open-Prompt-Injection
- id: r_bipia_base
  evaluation_id: ev_bipia_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '53.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, BIPIA-GPT (out of distribution)
- id: r_bipia_simpo
  evaluation_id: ev_bipia_simpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, BIPIA-GPT (out of distribution)
- id: r_bipia_sagec
  evaluation_id: ev_bipia_sagec
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, BIPIA-GPT (out of distribution)
- id: r_bipia_dpo
  evaluation_id: ev_bipia_dpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '7.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, BIPIA-GPT (out of distribution)
- id: r_bipia_orpo
  evaluation_id: ev_bipia_orpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '10.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, BIPIA-GPT (out of distribution)
- id: r_pieval_base
  evaluation_id: ev_pieval_base
  quantity: R1
  coverage: direct
  metric_as_printed: tool-result compliance %
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, LLM-PIEval
- id: r_pieval_simpo
  evaluation_id: ev_pieval_simpo
  quantity: R1
  coverage: direct
  metric_as_printed: tool-result compliance %
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, LLM-PIEval
- id: r_pieval_sagec
  evaluation_id: ev_pieval_sagec
  quantity: R1
  coverage: direct
  metric_as_printed: tool-result compliance %
  value: '1.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, LLM-PIEval
- id: r_pieval_dpo
  evaluation_id: ev_pieval_dpo
  quantity: R1
  coverage: direct
  metric_as_printed: tool-result compliance %
  value: '2.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, LLM-PIEval
- id: r_pieval_orpo
  evaluation_id: ev_pieval_orpo
  quantity: R1
  coverage: direct
  metric_as_printed: tool-result compliance %
  value: '1.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, LLM-PIEval
- id: r_scifact_base
  evaluation_id: ev_scifact_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '63.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, RIPE-II SciFact
- id: r_scifact_simpo
  evaluation_id: ev_scifact_simpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '8.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, RIPE-II SciFact
- id: r_scifact_sagec
  evaluation_id: ev_scifact_sagec
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '1.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, RIPE-II SciFact
- id: r_scifact_dpo
  evaluation_id: ev_scifact_dpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '2.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, RIPE-II SciFact
- id: r_scifact_orpo
  evaluation_id: ev_scifact_orpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '8.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 79
  location: Table 9.2, RIPE-II SciFact
- id: r_inj_base
  evaluation_id: ev_inj_base
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success (ASR-all)
  value: '0.303'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 85
  location: Table 9.3
- id: r_inj_simpo
  evaluation_id: ev_inj_simpo
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success (ASR-all)
  value: '0.468'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 85
  location: Table 9.3
- id: r_inj_sagec
  evaluation_id: ev_inj_sagec
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success (ASR-all)
  value: '0.533'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 85
  location: Table 9.3
- id: r_inj_dpo
  evaluation_id: ev_inj_dpo
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success (ASR-all)
  value: '0.605'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 85
  location: Table 9.3
- id: r_inj_orpo
  evaluation_id: ev_inj_orpo
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success (ASR-all)
  value: '0.017'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 85
  location: Table 9.3
- id: r_sc_stealth_8_asr
  evaluation_id: ev_sc_stealth_8
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.003'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 106
  location: Table 12.21, Stealth/obfusc. (in-dist)
- id: r_sc_stealth_8_det
  evaluation_id: ev_sc_stealth_8
  quantity: other
  coverage: direct
  metric_as_printed: Detect (truncation-aware)
  value: '0.950'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 106
  location: Table 12.21, Stealth/obfusc. (in-dist)
- id: r_sc_stealth_70_asr
  evaluation_id: ev_sc_stealth_70
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 106
  location: Table 12.21, Stealth/obfusc. (in-dist)
- id: r_sc_stealth_70_det
  evaluation_id: ev_sc_stealth_70
  quantity: other
  coverage: direct
  metric_as_printed: Detect (truncation-aware)
  value: '0.920'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 106
  location: Table 12.21, Stealth/obfusc. (in-dist)
- id: r_sc_bipiael_8_asr
  evaluation_id: ev_sc_bipiael_8
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 106
  location: Table 12.21, BIPIA-element (OOD)
- id: r_sc_bipiael_8_det
  evaluation_id: ev_sc_bipiael_8
  quantity: other
  coverage: direct
  metric_as_printed: Detect (truncation-aware)
  value: '1.000'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 106
  location: Table 12.21, BIPIA-element (OOD)
- id: r_sc_bipiael_70_asr
  evaluation_id: ev_sc_bipiael_70
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.040'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 106
  location: Table 12.21, BIPIA-element (OOD)
- id: r_sc_bipiael_70_det
  evaluation_id: ev_sc_bipiael_70
  quantity: other
  coverage: direct
  metric_as_printed: Detect (truncation-aware)
  value: '0.810'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 106
  location: Table 12.21, BIPIA-element (OOD)
- id: r_sc_bipiagpt_8_asr
  evaluation_id: ev_sc_bipiagpt_8
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 106
  location: Table 12.21, BIPIA-GPT (OOD)
- id: r_sc_bipiagpt_8_det
  evaluation_id: ev_sc_bipiagpt_8
  quantity: other
  coverage: direct
  metric_as_printed: Detect (truncation-aware)
  value: '0.707'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 106
  location: Table 12.21, BIPIA-GPT (OOD)
- id: r_sc_bipiagpt_70_asr
  evaluation_id: ev_sc_bipiagpt_70
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.007'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 106
  location: Table 12.21, BIPIA-GPT (OOD)
- id: r_sc_bipiagpt_70_det
  evaluation_id: ev_sc_bipiagpt_70
  quantity: other
  coverage: direct
  metric_as_printed: Detect (truncation-aware)
  value: '0.503'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 106
  location: Table 12.21, BIPIA-GPT (OOD)
comparisons:
- id: c_sc_stealth
  left: ev_sc_stealth_8
  right: ev_sc_stealth_70
  purpose: M
  c1_base: manipulated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the backbone size (8B against 70B), same data and objective
  page: 106
- id: c_sc_bipiael
  left: ev_sc_bipiael_8
  right: ev_sc_bipiael_70
  purpose: M
  c1_base: manipulated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the backbone size (8B against 70B), same data and objective
  page: 106
- id: c_sc_bipiagpt
  left: ev_sc_bipiagpt_8
  right: ev_sc_bipiagpt_70
  purpose: M
  c1_base: manipulated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the backbone size (8B against 70B), same data and objective
  page: 106
- id: c_t91s_sagec_base
  left: ev_t91s_sagec
  right: ev_t91s_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the preference training (SAGE-C against the same model under the same security prompt)
  page: 76
- id: c_t91s_sagec_simpo
  left: ev_t91s_sagec
  right: ev_t91s_simpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SAGE-C's adaptive margin, smoothing and weighting (against plain SimPO on the same pairs)
  page: 76
- id: c_t91s_sagec_dpo
  left: ev_t91s_sagec
  right: ev_t91s_dpo
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the training objective on the same pairs
  page: 76
- id: c_t91s_sagec_orpo
  left: ev_t91s_sagec
  right: ev_t91s_orpo
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the training objective on the same pairs
  page: 76
- id: c_t91c_sagec_base
  left: ev_t91c_sagec
  right: ev_t91c_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the preference training (SAGE-C against the same model under the same security prompt)
  page: 76
- id: c_t91c_sagec_simpo
  left: ev_t91c_sagec
  right: ev_t91c_simpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SAGE-C's adaptive margin, smoothing and weighting (against plain SimPO on the same pairs)
  page: 76
- id: c_t91c_sagec_dpo
  left: ev_t91c_sagec
  right: ev_t91c_dpo
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the training objective on the same pairs
  page: 76
- id: c_t91c_sagec_orpo
  left: ev_t91c_sagec
  right: ev_t91c_orpo
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the training objective on the same pairs
  page: 76
- id: c_cse3_sagec_base
  left: ev_cse3_sagec
  right: ev_cse3_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the preference training (SAGE-C against the same model under the same security prompt)
  page: 79
- id: c_cse3_sagec_simpo
  left: ev_cse3_sagec
  right: ev_cse3_simpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SAGE-C's adaptive margin, smoothing and weighting (against plain SimPO on the same pairs)
  page: 79
- id: c_opi_sagec_base
  left: ev_opi_sagec
  right: ev_opi_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the preference training (SAGE-C against the same model under the same security prompt)
  page: 79
- id: c_opi_sagec_simpo
  left: ev_opi_sagec
  right: ev_opi_simpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SAGE-C's adaptive margin, smoothing and weighting (against plain SimPO on the same pairs)
  page: 79
- id: c_bipia_sagec_base
  left: ev_bipia_sagec
  right: ev_bipia_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the preference training (SAGE-C against the same model under the same security prompt)
  page: 79
- id: c_bipia_sagec_simpo
  left: ev_bipia_sagec
  right: ev_bipia_simpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SAGE-C's adaptive margin, smoothing and weighting (against plain SimPO on the same pairs)
  page: 79
- id: c_pieval_sagec_base
  left: ev_pieval_sagec
  right: ev_pieval_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the preference training (SAGE-C against the same model under the same security prompt)
  page: 79
- id: c_pieval_sagec_simpo
  left: ev_pieval_sagec
  right: ev_pieval_simpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SAGE-C's adaptive margin, smoothing and weighting (against plain SimPO on the same pairs)
  page: 79
- id: c_scifact_sagec_base
  left: ev_scifact_sagec
  right: ev_scifact_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the preference training (SAGE-C against the same model under the same security prompt)
  page: 79
- id: c_scifact_sagec_simpo
  left: ev_scifact_sagec
  right: ev_scifact_simpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SAGE-C's adaptive margin, smoothing and weighting (against plain SimPO on the same pairs)
  page: 79
- id: c_inj_sagec_base
  left: ev_inj_sagec
  right: ev_inj_base
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the trained defence run outside its report format, against the undefended base model in the same native agent format
  page: 85
claims:
- id: k1
  text: On an 8B model that follows about 45% of stealth injections under the security prompt, GuardRAG cuts behavioural attack success to 0.3% while keeping benign-query utility
  location: abstract
  anchor: GuardRAG reduces behavioral attack success to 0.3% while preserving benign-query utility
  jobs:
  - R1
  - R2
  supported_by:
  - c_t91s_sagec_base
  - c_t91c_sagec_base
  page: 3
- id: k2
  text: The 8B model matches or exceeds the defence quality of a 70B model trained on the same compact data
  location: abstract
  anchor: matching or exceeding the defense quality of a 70B model
  jobs:
  - R1
  supported_by:
  - c_sc_stealth
  - c_sc_bipiael
  - c_sc_bipiagpt
  page: 3
- id: k3
  text: 'Placed in the native InjecAgent agent format, the trained defence does not help: SAGE-C''s attack success rises above the base model''s'
  location: Section 9.2.1
  anchor: the trained defense does not merely fail to help, it inverts
  jobs:
  - R1
  supported_by:
  - c_inj_sagec_base
  page: 84
coverage:
  R1: direct
  R2clean: proxy
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 9.1 (stealth split), Table 9.2 (five external benchmarks), Table 9.3 (native InjecAgent), Table 12.21
    R2clean: 'Table 9.1 utility retention on the clean split: a substantive answer with a parseable report and no attack flag; answer correctness is not scored'
    R2attack: not reported
    R3: not reported
schema_gaps:
- where: report
  note: the undefended reference in Tables 9.1 and 9.2 is the same 8B model under the identical staged security prompt (coded as baseline_kind instruction_prompt_only); in Table 9.3 the base model runs in InjecAgent's own
    format without that prompt
- where: defences:sagec
  note: trains_carry_through is coded yes_by_preference because the chosen responses copy an exact evidence span from the injected passage into the security report, and pairs prefer span-faithful reports over under-reporting
    ones
- where: claims:k2
  note: the claim speaks of defence quality, which the thesis reads mainly from detection; at equal size of data the 70B has the lower attack success on the stealth split (0.000 against 0.003) and the 8B on the two BIPIA splits
- where: report
  note: a master's thesis, not peer reviewed as a paper; preference candidates and judging use Llama-3.3-70B-Instruct and GPT-4o-mini; ORPO's low native-format value (0.017) is described by the author as a degenerate-output
    artifact

## Stage 2. The PDFs read are the PDFs registered, and every report is an included study

In [ ]:
import csv, logging, datetime as dt
from collections import Counter
import schema, search, corpus, paths
for m in (schema, search, corpus, paths):
    importlib.reload(m)
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader

BATCH = "04l"
cfg_exclusions = 0
cfg_next = "04_second_pass if it has not been run, then the re-code of the fields below the agreement threshold, then 04_full_extraction"
IDS = ['instrucot_arxiv2026', 'guardrag_uwthesis2026']
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
problems, texts = [], {}
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
    m = man.get(slug, {})
    if not pdf.exists():
        problems.append(f"{rid}: {pdf.name} not in data/defences"); continue
    sha = corpus.sha256(pdf)
    rd = PdfReader(str(pdf))
    texts[slug] = ([p.extract_text() or "" for p in rd.pages], [p.extract_text(extraction_mode="layout") or "" for p in rd.pages])
    ok_sha = sha == m.get("sha256")
    ok_inc = m.get("decision") == "include"
    print(f"{rid:<28} {slug:<12} pages {len(rd.pages):>3}  sha256 {'matches' if ok_sha else 'DIFFERS'}  decision {m.get('decision')}")
    if not ok_sha:
        problems.append(f"{rid}: PDF differs from the one registered in reports/defences.csv")
    if not ok_inc:
        problems.append(f"{rid}: not an included study")

## Stage 3. Validate against the frozen schema and the PDFs; grade the comparisons; extra checks

In [ ]:
import re
import pypdfium2 as pdfium
_docs = {}
def pdfium_page(slug, page):
    if slug not in _docs:
        _docs[slug] = pdfium.PdfDocument(str(paths.DATA_DEFENCES / f"{slug}.pdf"))
    return _docs[slug][page - 1].get_textpage().get_text_range()
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
check_rows, summary, residual, third_rows = [], [], [], []
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    plain, lay = texts.get(slug, (None, None))
    probs, notes = schema.validate_record(rec, S1, plain, lay)
    kept, third = [], []                       # a value pypdf glues to its neighbour is looked for in a third extraction (pypdfium2)
    for p in probs:
        m = re.match(r"results:(\S+): value '(.+)' not found on page (\d+)$", p)
        if m and schema.value_on_page(m.group(2), pdfium_page(slug, int(m.group(3)))):
            third.append(f"{m.group(1)} ({m.group(2)!r}, page {m.group(3)})")
        else:
            kept.append(p)
    probs = kept
    for kind in ("defences", "evaluations", "results", "comparisons", "claims"):          # ids unique within each kind
        dup = [i for i, n in Counter(r["id"] for r in rec.get(kind) or []).items() if n > 1]
        probs += [f"{kind}: id {i} used more than once" for i in dup]
    fig_kept, third_chart = 0, []
    for r in rec.get("results") or []:                                                     # chart values kept as printed
        if r.get("value_source") == "figure_only" and r.get("value") not in (None, ""):
            fig_kept += 1
            pg = int(r["page"]) - 1
            if plain and not (schema.value_on_page(r["value"], plain[pg]) or schema.value_on_page(r["value"], lay[pg])):
                if schema.value_on_page(r["value"], pdfium_page(slug, int(r["page"]))):
                    third_chart.append(f"{r['id']} ({r['value']!r}, page {r['page']}, chart label)")
                else:
                    probs.append(f"results:{r['id']}: chart value {r['value']!r} not found on page {r['page']}")
    third_rows += [{"report": rid, "status": "value found in the third extraction", "detail": t} for t in third + third_chart]
    grades = Counter(schema.grade(c) for c in rec.get("comparisons") or [])
    n_val = sum(1 for r in rec.get("results") or [] if r.get("value_source") != "figure_only")
    n_anc = sum(1 for k in ("defences", "claims") for r in rec.get(k) or [] if r.get("anchor"))
    summary.append({"batch": BATCH, "report": rid, "slug": slug, "role": rec["report"]["role"],
                    "defences": len(rec.get("defences") or []), "evaluations": len(rec.get("evaluations") or []),
                    "results": len(rec.get("results") or []), "values_checked": n_val, "chart_values_checked": fig_kept,
                    "anchors_checked": n_anc, "comparisons": len(rec.get("comparisons") or []),
                    "grade_A": grades["A"], "grade_B": grades["B"], "grade_C": grades["C"], "grade_U": grades["U"],
                    "claims": len(rec.get("claims") or []),
                    **{f"cov_{q}": rec["coverage"][q] for q in ("R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe")},
                    "values_third_extraction": len(third), "chart_values_third_extraction": len(third_chart), "residual_notes": len(notes), "problems": len(probs)})
    check_rows += [{"report": rid, "status": "problem", "detail": p} for p in probs]
    residual += [{"report": rid, "where": n.split(": ", 1)[0], "note": n.split(": ", 1)[1]} for n in notes]
    print(f"{rid:<28} problems {len(probs)}; values {n_val} ({len(third)} via the third extraction); chart values {fig_kept} ({len(third_chart)} via the third extraction); anchors {n_anc}; grades {dict(grades)}")
    for p in probs:
        print("   ", p)
    problems += [f"{rid}: {p}" for p in probs]
check_rows += third_rows + [{"report": g["report"], "status": "residual note", "detail": f"{g['where']}: {g['note']}"} for g in residual]
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_check.csv", check_rows or [{"report": "all", "status": "ok", "detail": "no problems"}],
                 ["report", "status", "detail"])
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_summary.csv", summary, list(summary[0].keys()))
for g in residual:
    print(f"  residual note, {g['report']}: {g['note'][:150]}")

## Stage 4. Batch register (one row per coded report, kept across batches)

In [ ]:
reg_path = paths.REPORTS / "extraction_batches.csv"
reg = [r for r in search.read_csv(reg_path) if r["batch"] != BATCH] if reg_path.exists() else []
today = dt.date.today().isoformat()
for s in summary:
    reg.append({"batch": BATCH, "report": s["report"], "slug": s["slug"], "coded": today,
                "status": "checked" if s["problems"] == 0 else "problems"})
reg.sort(key=lambda r: (r["batch"], r["report"]))
search.write_csv(reg_path, reg, ["batch", "report", "slug", "coded", "status"])
included = sum(1 for m in search.read_csv(paths.REPORTS / "defences.csv") if m.get("decision") == "include")   # read again: Stage 2b may change it
pilot = len(list(paths.RECORDS.glob("*.yaml"))) - len({r["report"] for r in reg})
print(f"records in reports/records: {len(list(paths.RECORDS.glob('*.yaml')))} (pilot {pilot}, batches {len({r['report'] for r in reg})}); included studies: {included}")

## Stage 5. Decision record and tests

In [ ]:
tot = {k: sum(s[k] for s in summary) for k in ("defences", "evaluations", "results", "values_checked", "values_third_extraction", "chart_values_checked",
                                              "chart_values_third_extraction", "anchors_checked", "comparisons", "grade_A", "grade_B", "grade_C", "grade_U", "claims")}
status = "BATCH CHECKED" if not problems else "BATCH NOT CHECKED: PROBLEMS BELOW"
lines = [f"decision_{BATCH}: {status}", f"this step: {dt.date.today().isoformat()}",
         "reports: " + ", ".join(IDS),
         f"records: {tot['defences']} defences, {tot['evaluations']} evaluations, {tot['results']} results, "
         f"{tot['comparisons']} comparisons, {tot['claims']} claims",
         f"checked against the PDFs: {tot['values_checked']} printed values ({tot['values_third_extraction']} of them found only in the third "
         f"extraction), {tot['chart_values_checked']} chart values ({tot['chart_values_third_extraction']} of them found only in the third extraction) "
         f"and {tot['anchors_checked']} anchors, each found on its page",
         f"comparison grades: A {tot['grade_A']}, B {tot['grade_B']}, C {tot['grade_C']}, U {tot['grade_U']}",
         f"residual notes: {len(residual)}; full-text exclusions found while coding: {cfg_exclusions}",
         f"next: {cfg_next}"] + [f"- problem: {p}" for p in problems]
(paths.REPORTS / f"decision_{BATCH}.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 6. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04l extraction batch l: the last two included studies coded from their PDFs, values and anchors checked, comparisons graded"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)